In [1]:
import torch 
import torch.nn as nn
import torch.nn.functional as F

In [2]:
# Shape: (batch, heads, sequence_length, head_dim)
B, H, N, D = 2, 8, 5, 64

q = torch.randn(B, H, 100, D)
k = torch.randn(B, H, 5, D)
v = torch.randn(B, H, 5, 10)

valid = torch.tensor([
    [1, 0, 1, 1, 0],  # Batch 1
    [1, 1, 0, 0, 1]   # Batch 2
], dtype=torch.bool)

mask = valid[:, None, None, :]

out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)

print(out.shape)
# torch.Size([2, 8, 128, 64])

torch.Size([2, 8, 100, 10])


In [3]:
mask

tensor([[[[ True, False,  True,  True, False]]],


        [[[ True,  True, False, False,  True]]]])

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ExternalPerturbAttention(nn.Module):

    def __init__(self, dim, num_heads=8, dropout=0.1):
        super().__init__()

        assert dim % num_heads == 0

        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.dropout = dropout

        # Part 1
        self.kv_linear = nn.Linear(dim, dim * 2)
        self.q_linear = nn.Linear(dim, dim)

        self.chem_ffn = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Linear(dim * 4, dim),
        )

        # Part 2
        self.kqv = nn.Linear(dim * 2, dim * 3)

        self.gene_ffn = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Linear(dim * 4, dim),
        )

    def split_heads(self, x):
        B, T, C = x.shape

        x = x.view(B, T, self.num_heads, self.head_dim)

        return x.transpose(1, 2)

    def merge_heads(self, x):
        B, H, T, D = x.shape

        return x.transpose(1, 2).contiguous().view(
            B, T, H * D
        )

    def forward(self, x, chem, mask=None):

        # Part 1
        kv = self.kv_linear(chem)
        k, v = kv.chunk(2, dim=-1)

        q = self.q_linear(x)

        q = self.split_heads(q)
        k = self.split_heads(k)
        v = self.split_heads(v)

        out1 = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=mask,
            dropout_p=self.dropout if self.training else 0.0
        )

        out1 = self.merge_heads(out1)

        out1 = self.chem_ffn(out1)

        # Part 2
        kqv = self.kqv(torch.cat([x, out1], dim=-1))

        k, q, v = kqv.chunk(3, dim=-1)

        k = self.split_heads(k)
        q = self.split_heads(q)
        v = self.split_heads(v)

        out2 = F.scaled_dot_product_attention(
            q, k, v,
            dropout_p=self.dropout if self.training else 0.0
        )

        out2 = self.merge_heads(out2)

        out2 = self.gene_ffn(out2)
        
        out = out2 + x

        return out

In [ ]:
class pertubationmodel(nn.Module):
    def __init__(self, dim = 1024, conc_dim = 128, num_heads=8, dropout=0.1, num_layers=4, num_of_genes=1000,chem_dim=128, cell_coordi_dim = 2):
        super().__init__()
        
        self.num_gene = num_of_genes
        
        self.gene_embedding = nn.Embedding(num_of_genes, dim)
        
        self.chem_embedding = nn.Linear(chem_dim, dim)
        
        self.layers = nn.ModuleList([
            ExternalPerturbAttention(
                dim=dim,
                num_heads=num_heads,
                dropout=dropout
            )
            for _ in range(num_layers)
        ])
        
        self.conc_emb = nn.Linear(dim, conc_dim)
        
        self.conc_ffn = nn.Sequential(
            nn.Linear(conc_dim, dim * 2),
            nn.GELU(),
            nn.Linear(dim * 2, conc_dim),
        )
        
        self.merger = nn.Linear(dim + conc_dim, dim)
        
        self.coordi_layer = nn.Linear(dim + cell_coordi_dim, 1)
        
        self.coordi_layer = nn.Sequential(
            nn.Linear(dim + cell_coordi_dim, dim // 2),
            nn.GELU(),
            nn.Linear(dim // 2, dim // 4),
            nn.GELU(),
            nn.Linear(dim // 4, 1)
        )
        
    def forward(self, control_cell_line, chem, cell_coordi, mask=None):

        gene_norms = torch.linalg.norm(control_cell_line,axis = -1, keepdim=True)
        control_cell_line = control_cell_line / gene_norms.clamp_min(1e-12)
        
        B = control_cell_line.shape[0]
        
        gene_emb = self.gene_embedding(torch.arange(self.num_gene)).unsqueeze(0).expand(B, -1, -1)
        
        print("Gene embedding shape:", gene_emb.shape)  # Debugging line
        
        print("Chemical concentration shape:", chem.shape)  # Debugging line
        
        chem_emb = self.chem_embedding(chem)
        
        print("Chemical embedding shape:", chem_emb.shape)  # Debugging line
        
        #part1 # this part take the control cell line and mix the information with the emb

        concise_emb = self.conc_emb(gene_emb)
        
        print("Concise embedding shape:", concise_emb.shape)  # Debugging line
        print("Control cell line shape:", control_cell_line.shape)  # Debugging line
        
        out_conc = F.scaled_dot_product_attention(control_cell_line, control_cell_line, concise_emb, scale = 10, dropout_p=0)
        
        print("Output concentration shape:", out_conc.shape)  # Debugging line
        
        out_conc = self.conc_ffn(out_conc)
        
        print("Output concentration after FFN shape:", out_conc.shape)  # Debugging line
        
        cell_state = self.merger(torch.cat([gene_emb, out_conc], dim=-1))
        
        print("Cell state shape after merger:", cell_state.shape)  # Debugging line
        print("mask shape:" , mask.shape if mask is not None else "No mask")  # Debugging line
        print("Chemical embedding shape:", chem_emb.shape)  # Debugging line
        #this part mix the cell state with chemical information
        
        for layer in self.layers:
            cell_state = layer(cell_state, chem_emb, mask)
            
        #part2 # this part take the cell state and mix the information with the cell coordinates
        print("Cell state shape before coordinate processing:", cell_state.shape)  # Debugging line
        print("Cell coordinates shape:", cell_coordi.shape)  # Debugging line
        
        
        cellcoordi = cell_coordi.unsqueeze(2).expand(-1, -1, cell_state.shape[1], -1)
        cell_state_expanded = cell_state.unsqueeze(1).expand(-1, cell_coordi.shape[1], -1, -1)
        print("Cell coordinates expanded shape:", cellcoordi.shape)  # Debugging line
        print("Cell state expanded shape:", cell_state_expanded.shape)  # Debugging line
        combined = torch.cat([cellcoordi, cell_state_expanded], dim=-1)
        print("Combined shape:", combined.shape)  # Debugging line
        output = self.coordi_layer(combined).squeeze(-1)
        print("Output shape:", output.shape)  # Debugging line
        return output

In [6]:
model = pertubationmodel(
            dim = 256,
            conc_dim = 32,
            num_heads=4,
            num_layers=2,
            num_of_genes=1000,
            chem_dim=129,
            cell_coordi_dim = 2
        )

In [7]:
X_control = torch.randn(2,1000,300) # batch , number_of_genes, num_of_cells
X_chem = torch.randn(2,10,128) # batch, chem_dim
X_chem_conc = torch.randn(2,1) # batch, chem_conc_dim
X_chem_mask = torch.tensor([
    [1, 0, 1, 1, 0, 1, 0, 1, 1, 0],  # Batch 1
    [1, 1, 0, 0, 1, 1, 0, 1, 1, 0]   # Batch 2
], dtype=torch.bool)[:, None, None, :]
X_chem_conc = X_chem_conc.unsqueeze(1).expand(-1, 10, -1)  # (2, 10, 1)
X_chem = torch.cat([X_chem, X_chem_conc], dim=-1)
print("X_chem shape:", X_chem.shape)  # Debugging line
y_coordi = torch.randn(2, 5, 2) # batch, num_of_cells, cell_coordi_dim

X_chem shape: torch.Size([2, 10, 129])


In [8]:
out = model(X_control, X_chem, y_coordi, X_chem_mask)

Gene embedding shape: torch.Size([2, 1000, 256])
Chemical concentration shape: torch.Size([2, 10, 129])
Chemical embedding shape: torch.Size([2, 10, 256])
Concise embedding shape: torch.Size([2, 1000, 32])
Control cell line shape: torch.Size([2, 1000, 300])
Output concentration shape: torch.Size([2, 1000, 32])
Output concentration after FFN shape: torch.Size([2, 1000, 32])
Cell state shape after merger: torch.Size([2, 1000, 256])
mask shape: torch.Size([2, 1, 1, 10])
Chemical embedding shape: torch.Size([2, 10, 256])
Cell state shape before coordinate processing: torch.Size([2, 1000, 256])
Cell coordinates shape: torch.Size([2, 5, 2])
Cell coordinates expanded shape: torch.Size([2, 5, 1000, 2])
Cell state expanded shape: torch.Size([2, 5, 1000, 256])
Combined shape: torch.Size([2, 5, 1000, 258])
Output shape: torch.Size([2, 5, 1000])


In [19]:
import torch
import math

def perturbation_loss(x, y,z = 0.5):
    num_genes = x.shape[-1]
    loss = (y - x) ** 2
    loss = loss.sum(dim=-1)   # Sum across genes
    loss = loss.mean(dim=-1)  # Mean across cells
    loss = loss.mean()        # Mean across batch

    return loss / (num_genes**z)


# Example
x = torch.ones(2, 100, 1000, requires_grad=True)
y = torch.zeros(2, 100, 1000)

loss = perturbation_loss(x, y, 0.75)

print("Loss:", loss.item())
print("Loss shape:", loss.shape)
print("MSE Loss:", torch.nn.functional.mse_loss(x, y).item())
# Backpropagation
loss.backward()

Loss: 5.6234130859375
Loss shape: torch.Size([])
MSE Loss: 1.0
